# Cadenas · Retos de cadenas

Retos cortos sobre cadenas: traductores (Morse, *leet*, T9, un cifrado por sustitución), clasificación de palabras (heterogramas, isogramas, pangramas), conversiones (columnas de Excel), análisis de texto (URL, contraseñas, expresiones) y formato de salida. Cada función se comprueba con `assert`, a menudo contra la biblioteca estándar o una segunda solución. Ideas de [retosdeprogramacion.com](https://retosdeprogramacion.com/ejercicios) y [PYnative](https://pynative.com/python-exercises-with-solutions/).

## 1. Traductor de Morse en los dos sentidos

Escribir `traducir_morse(texto)`, que detecte si recibe texto normal o Morse y lo traduzca al otro formato. En Morse las letras se separan con un espacio y las palabras con ` / `. Las tildes se quitan antes de traducir y la `Ñ` tiene su propio código (`--.--`). Comprobar que el viaje de ida y vuelta devuelve el texto en mayúsculas y sin tildes.

In [1]:
import random
import re
import string
from collections import Counter
from itertools import product, zip_longest
from urllib.parse import parse_qs

SIN_TILDES = str.maketrans("áéíóúüÁÉÍÓÚÜ", "aeiouuAEIOUU")

MORSE = {
    "A": ".-", "B": "-...", "C": "-.-.", "D": "-..", "E": ".", "F": "..-.", "G": "--.", "H": "....",
    "I": "..", "J": ".---", "K": "-.-", "L": ".-..", "M": "--", "N": "-.", "Ñ": "--.--", "O": "---",
    "P": ".--.", "Q": "--.-", "R": ".-.", "S": "...", "T": "-", "U": "..-", "V": "...-", "W": ".--",
    "X": "-..-", "Y": "-.--", "Z": "--..", "0": "-----", "1": ".----", "2": "..---", "3": "...--",
    "4": "....-", "5": ".....", "6": "-....", "7": "--...", "8": "---..", "9": "----.",
    ".": ".-.-.-", ",": "--..--", "?": "..--..", "!": "-.-.--",
}
DESDE_MORSE = {codigo: letra for letra, codigo in MORSE.items()}


def es_morse(texto):
    return set(texto) <= set(".-/ ")


def traducir_morse(texto):
    if es_morse(texto):
        palabras = texto.strip().split(" / ")
        return " ".join("".join(DESDE_MORSE[c] for c in p.split()) for p in palabras)
    palabras = texto.translate(SIN_TILDES).upper().split()
    return " / ".join(" ".join(MORSE[c] for c in p) for p in palabras)


frase = "Mañana subimos al Teide, ¿vienes?"
frase = frase.replace("¿", "")  # el símbolo de apertura no existe en Morse
codigo = traducir_morse(frase)
print(codigo)
print(traducir_morse(codigo))

assert traducir_morse("SOS") == "... --- ..."
assert traducir_morse("... --- ...") == "SOS"
for t in ["Hola desde Las Palmas", "Vuelo 1234 a El Hierro.", "La cigüeña y el búho"]:
    assert traducir_morse(traducir_morse(t)) == t.translate(SIN_TILDES).upper()

-- .- --.-- .- -. .- / ... ..- -... .. -- --- ... / .- .-.. / - . .. -.. . --..-- / ...- .. . -. . ... ..--..
MAÑANA SUBIMOS AL TEIDE, VIENES?


## 2. *Leet* y un cifrado por sustitución

Escribir `a_leet(texto)` con una tabla propia que cambie algunas letras por dígitos parecidos (`a→4`, `e→3`, `o→0`…) y `desde_leet`, que deshaga el cambio en textos sin dígitos. Después, crear un cifrado en cuatro pasos: invertir la palabra, cambiar las vocales por números (`a→0`, `e→1`, `i→2`, `o→3`, `u→4`) y añadir el sufijo `gc`. Escribir también el descifrado y comprobar el viaje de ida y vuelta.

In [2]:
LEET = str.maketrans("abegiostz", "483610572")
DESDE_LEET = str.maketrans("483610572", "abegiostz")


def a_leet(texto):
    return texto.lower().translate(LEET)


def desde_leet(texto):
    return texto.translate(DESDE_LEET)


for t in ["Las Palmas de Gran Canaria", "programar es divertido"]:
    print(f"{t!r} → {a_leet(t)!r}")
    assert desde_leet(a_leet(t)) == t.lower()

'Las Palmas de Gran Canaria' → 'l45 p4lm45 d3 6r4n c4n4r14'
'programar es divertido' → 'pr06r4m4r 35 d1v3r71d0'


In [3]:
VOCAL_A_NUMERO = str.maketrans("aeiou", "01234")
NUMERO_A_VOCAL = str.maketrans("01234", "aeiou")
SUFIJO = "gc"


def cifrar(palabra):
    return palabra.lower()[::-1].translate(VOCAL_A_NUMERO) + SUFIJO


def descifrar(cifrada):
    if not cifrada.endswith(SUFIJO):
        raise ValueError(f"{cifrada!r} no termina en {SUFIJO!r}")
    return cifrada.removesuffix(SUFIJO).translate(NUMERO_A_VOCAL)[::-1]


for palabra in ["palmera", "guagua", "barranco", "tunera"]:
    c = cifrar(palabra)
    print(f"{palabra:<10} → {c}")
    assert descifrar(c) == palabra
assert cifrar("palmera") == "0r1ml0pgc"
try:
    descifrar("0r1ml0p")
except ValueError as e:
    print("Error:", e)

palmera    → 0r1ml0pgc
guagua     → 04g04ggc
barranco   → 3cn0rr0bgc
tunera     → 0r1n4tgc
Error: '0r1ml0p' no termina en 'gc'


## 3. Vocal más frecuente y caracteres desparejados

Escribir tres funciones:

- `vocales_mas_frecuentes(texto)`: devuelve la lista ordenada de las vocales que más se repiten, sin distinguir mayúsculas ni tildes (puede haber empate).
- `desparejados(a, b)`: devuelve los caracteres de `a` que no están en `b` y los de `b` que no están en `a`, sin repetir y en orden de aparición.
- `caracter_colado(original, modificada)`: `modificada` es `original` desordenada con un carácter de más; encontrarlo.

In [4]:
def vocales_mas_frecuentes(texto):
    cuenta = Counter(c for c in texto.lower().translate(SIN_TILDES) if c in "aeiou")
    if not cuenta:
        return []
    maximo = max(cuenta.values())
    return sorted(v for v, n in cuenta.items() if n == maximo)


def vocales_fuerza_bruta(texto):
    limpio = texto.lower().translate(SIN_TILDES)
    veces = {v: limpio.count(v) for v in "aeiou"}
    maximo = max(veces.values())
    return [v for v in "aeiou" if veces[v] == maximo and maximo > 0]


for t in ["Murciélago", "La guagua llegó tarde a Arucas", "Él es un ejemplo", "xyz"]:
    print(f"{t!r}: {vocales_mas_frecuentes(t)}")
    assert vocales_mas_frecuentes(t) == vocales_fuerza_bruta(t)

'Murciélago': ['a', 'e', 'i', 'o', 'u']
'La guagua llegó tarde a Arucas': ['a']
'Él es un ejemplo': ['e']
'xyz': []


In [5]:
def desparejados(a, b):
    solo_a = "".join(dict.fromkeys(c for c in a if c not in b))
    solo_b = "".join(dict.fromkeys(c for c in b if c not in a))
    return solo_a, solo_b


def caracter_colado(original, modificada):
    (colado,) = (Counter(modificada) - Counter(original)).elements()
    return colado


print(desparejados("teide", "timple"))
assert desparejados("teide", "timple") == ("d", "mpl")
assert set("".join(desparejados("teide", "timple"))) == set("teide") ^ set("timple")

random.seed(7)
for _ in range(200):
    original = "".join(random.choices("abcdeñ", k=random.randint(0, 10)))
    extra = random.choice("abcdeñ")
    modificada = list(original + extra)
    random.shuffle(modificada)
    modificada = "".join(modificada)
    assert caracter_colado(original, modificada) == extra
    # Alternativa: la diferencia de las sumas de los códigos
    assert chr(sum(map(ord, modificada)) - sum(map(ord, original))) == extra
print(caracter_colado("gofio", "fogiox"))

('d', 'mpl')
x


## 4. Heterograma, isograma y pangrama

Contando solo letras (sin distinguir mayúsculas ni tildes, pero con la `ñ` como letra propia), escribir:

- `es_heterograma(texto)`: ninguna letra se repite.
- `es_isograma(texto)`: todas las letras que aparecen lo hacen el mismo número de veces (devolver ese número o 0 si no lo es).
- `es_pangrama(texto)`: aparecen las 27 letras del alfabeto español. Escribir también `letras_que_faltan`.

In [6]:
ALFABETO = "abcdefghijklmnñopqrstuvwxyz"


def letras(texto):
    return [c for c in texto.lower().translate(SIN_TILDES) if c in ALFABETO]


def es_heterograma(texto):
    ls = letras(texto)
    return len(ls) == len(set(ls))


def es_isograma(texto):
    veces = set(Counter(letras(texto)).values())
    return veces.pop() if len(veces) == 1 else 0


def letras_que_faltan(texto):
    return "".join(sorted(set(ALFABETO) - set(letras(texto)), key=ALFABETO.index))


def es_pangrama(texto):
    return not letras_que_faltan(texto)


for t in ["Murciélago", "Guanche", "cucú", "papá", "Tenerife", "Roque Nublo"]:
    print(f"{t:<12} heterograma={es_heterograma(t)!s:<5} isograma={es_isograma(t)}")

assert es_heterograma("Murciélago") and not es_heterograma("Tenerife")
assert es_isograma("cucú") == 2 and es_isograma("Guanche") == 1 and es_isograma("Tenerife") == 0
# Todo heterograma es un isograma de orden 1
assert all(es_isograma(t) == 1 for t in ["Murciélago", "Guanche", "Roque Nublo"] if es_heterograma(t))

Murciélago   heterograma=True  isograma=1
Guanche      heterograma=True  isograma=1
cucú         heterograma=False isograma=2
papá         heterograma=False isograma=2
Tenerife     heterograma=False isograma=0
Roque Nublo  heterograma=False isograma=0


In [7]:
pangrama = "Félix, el guanche, vuelve hoy de Jandía con queso, kiwi, ñame y zumo de tuno para brindar."
casi = "Félix, el guanche, vuelve hoy de Jandía con queso, kiwi y zumo para brindar."
print(es_pangrama(pangrama), es_pangrama(casi), repr(letras_que_faltan(casi)))
assert es_pangrama(pangrama)
assert letras_que_faltan(casi) == "ñt"
assert not es_pangrama(pangrama.replace("ñ", "n"))

True False 'ñt'


## 5. Palabras que valen 100

Cada letra vale su posición en el alfabeto español: `A = 1`, …, `N = 14`, `Ñ = 15`, `O = 16`, …, `Z = 27`. Escribir `puntuacion(palabra)` y buscar en una lista de palabras las que suman exactamente 100. Mostrar también las tres más cercanas a 100 que no llegan a sumarlo.

In [8]:
VALOR = {letra: i for i, letra in enumerate(ALFABETO, start=1)}


def puntuacion(palabra):
    return sum(VALOR[c] for c in letras(palabra))


palabras = """guagua gofio almogrote timple barraquito tajinaste sancocho tenderete volcanes arrugadas
comparsa tsunami guitarra tomates pepinos zanahoria asteroide caminantes gaviotas mostaza tomillo
diccionario ordenador lagartos senderos""".split()

assert puntuacion("abc") == 6 and puntuacion("ñ") == 15 and puntuacion("Z") == 27
assert puntuacion("Ñandú") == puntuacion("ñandu")

cien = [p for p in palabras if puntuacion(p) == 100]
print("Suman 100:", cien)
cercanas = sorted((p for p in palabras if puntuacion(p) != 100), key=lambda p: abs(puntuacion(p) - 100))[:3]
for p in cercanas:
    print(f"  {p:<12} {puntuacion(p)}")
# Comprobación con otra forma de calcular el valor
assert cien == [p for p in palabras if sum(ALFABETO.index(c) + 1 for c in p) == 100]
assert cien == ["tsunami", "asteroide"]

Suman 100: ['tsunami', 'asteroide']
  tenderete    99
  guitarra     99
  caminantes   101


## 6. Columnas de una hoja de cálculo

Las columnas de una hoja de cálculo se nombran `A`, `B`, …, `Z`, `AA`, `AB`, …, `AZ`, `BA`, …, `ZZ`, `AAA`… Escribir `columna_a_numero(nombre)` y `numero_a_columna(n)` y comprobarlos con `A = 1`, `Z = 26`, `AA = 27`, `AZ = 52`, `ZZ = 702` y `AAA = 703`, con el viaje de ida y vuelta y contra una enumeración generada con `itertools.product`.

In [9]:
LETRAS = string.ascii_uppercase


def columna_a_numero(nombre):
    n = 0
    for c in nombre:
        n = n * 26 + LETRAS.index(c) + 1
    return n


def numero_a_columna(n):
    if n < 1:
        raise ValueError("las columnas empiezan en 1")
    nombre = ""
    while n:
        n, resto = divmod(n - 1, 26)  # base 26 sin cero: A..Z son 1..26
        nombre = LETRAS[resto] + nombre
    return nombre


ejemplos = {"A": 1, "Z": 26, "AA": 27, "AZ": 52, "BA": 53, "ZZ": 702, "AAA": 703, "XFD": 16384}
for nombre, n in ejemplos.items():
    print(f"{nombre:>4} ↔ {n}")
    assert columna_a_numero(nombre) == n and numero_a_columna(n) == nombre

enumeradas = ["".join(t) for k in (1, 2, 3) for t in product(LETRAS, repeat=k)]
assert all(numero_a_columna(i) == nombre for i, nombre in enumerate(enumeradas, start=1))
assert all(columna_a_numero(numero_a_columna(n)) == n for n in range(1, 20_000))

   A ↔ 1
   Z ↔ 26
  AA ↔ 27
  AZ ↔ 52
  BA ↔ 53
  ZZ ↔ 702
 AAA ↔ 703
 XFD ↔ 16384


## 7. Teclado T9

En los teléfonos antiguos cada tecla tenía varias letras (`2 → abc`, `3 → def`, …, `7 → pqrs`, `9 → wxyz`) y el `0` era el espacio. Una letra se escribía pulsando su tecla tantas veces como su posición. Escribir `decodificar_t9`, que reciba los grupos de pulsaciones separados por guiones (`44-666-555-2` → `hola`), y `codificar_t9`. Un grupo con teclas distintas es un error.

In [10]:
TECLAS = {"2": "abc", "3": "def", "4": "ghi", "5": "jkl", "6": "mno", "7": "pqrs", "8": "tuv", "9": "wxyz", "0": " "}
A_PULSACIONES = {letra: tecla * i for tecla, ls in TECLAS.items() for i, letra in enumerate(ls, start=1)}


def decodificar_t9(secuencia):
    texto = []
    for grupo in secuencia.split("-"):
        if len(set(grupo)) != 1 or grupo[0] not in TECLAS:
            raise ValueError(f"grupo no válido: {grupo!r}")
        ls = TECLAS[grupo[0]]
        texto.append(ls[(len(grupo) - 1) % len(ls)])  # al pasarse se da la vuelta
    return "".join(texto)


def codificar_t9(texto):
    return "-".join(A_PULSACIONES[c] for c in texto.lower())


print(decodificar_t9("44-666-555-2"))
print(codificar_t9("ven a la playa"))
assert decodificar_t9("44-666-555-2") == "hola"
assert decodificar_t9("2222") == "a"
for t in ["ven a la playa", "quedamos en teror", "xyz"]:
    assert decodificar_t9(codificar_t9(t)) == t
try:
    decodificar_t9("44-656")
except ValueError as e:
    print("Error:", e)

hola
888-33-66-0-2-0-555-2-0-7-555-2-999-2
Error: grupo no válido: '656'


## 8. Parámetros de una URL

Extraer a mano (sin `urllib`) los parámetros de la consulta de una URL: lo que va entre `?` y `#`, separado por `&`, con pares `clave=valor`. Un `+` es un espacio y `%XX` es un byte en hexadecimal (los caracteres no ASCII ocupan varios bytes en UTF-8). Como una clave puede repetirse, devolver un diccionario `clave → lista de valores` y comprobarlo contra `urllib.parse.parse_qs`.

In [11]:
def decodificar_porcentajes(texto):
    datos = texto.replace("+", " ").encode()
    salida = bytearray()
    i = 0
    while i < len(datos):
        if datos[i] == ord("%"):
            salida.append(int(datos[i + 1:i + 3], 16))
            i += 3
        else:
            salida.append(datos[i])
            i += 1
    return salida.decode()


def parametros(url):
    consulta = url.split("#", 1)[0].partition("?")[2]
    resultado = {}
    for par in consulta.split("&"):
        if not par:
            continue
        clave, _, valor = par.partition("=")
        resultado.setdefault(decodificar_porcentajes(clave), []).append(decodificar_porcentajes(valor))
    return resultado


url = ("https://ejemplo.org/rutas?isla=la+palma&inicio=Los%20Llanos&destino=Ca%C3%B1adas"
       "&etiqueta=volcan&etiqueta=pinar&vacio=#mapa")
for clave, valores in parametros(url).items():
    print(f"{clave:<9} {valores}")

consulta = url.split("#")[0].split("?")[1]
assert parametros(url) == parse_qs(consulta, keep_blank_values=True)
assert parametros("https://ejemplo.org/") == {}

isla      ['la palma']
inicio    ['Los Llanos']
destino   ['Cañadas']
etiqueta  ['volcan', 'pinar']
vacio     ['']


## 9. Palabras en un marco

Escribir `enmarcar(frase)`, que devuelva una cadena con cada palabra de la frase en una línea, alineada a la izquierda dentro de un marco de asteriscos con un espacio de margen a cada lado. Devolver la cadena (en lugar de imprimirla) permite comprobar el resultado con `assert`.

In [12]:
def enmarcar(frase):
    palabras = frase.split()
    ancho = max(map(len, palabras), default=0)
    borde = "*" * (ancho + 4)
    return "\n".join([borde, *(f"* {p:<{ancho}} *" for p in palabras), borde])


print(enmarcar("¿Qué es la vida sin un barraquito?"))
assert enmarcar("hola isla") == "********\n* hola *\n* isla *\n********"
lineas = enmarcar("una frase con palabras de longitudes distintas").splitlines()
assert len({len(l) for l in lineas}) == 1 and set(lineas[0]) == set(lineas[-1]) == {"*"}

***************
* ¿Qué        *
* es          *
* la          *
* vida        *
* sin         *
* un          *
* barraquito? *
***************


## 10. Centros, minúsculas primero e intercalados

- Devolver los tres caracteres centrales de una cadena de longitud impar (al menos 3) e insertar una cadena en el centro de otra.
- Reordenar una cadena para que las minúsculas vayan antes que las mayúsculas, manteniendo el orden relativo dentro de cada grupo.
- Intercalar dos cadenas carácter a carácter (`abc`, `12345` → `a1b2c345`) y, en otra versión, tomando uno del principio de la primera y otro del final de la segunda.

In [13]:
def tres_centrales(s):
    if len(s) < 3 or len(s) % 2 == 0:
        raise ValueError(f"{s!r} no tiene longitud impar de al menos 3")
    medio = len(s) // 2
    return s[medio - 1:medio + 2]


def insertar_en_centro(s, otra):
    medio = len(s) // 2
    return s[:medio] + otra + s[medio:]


print(tres_centrales("Teror"), tres_centrales("Tazacorte"), insertar_en_centro("cuatro", "--"))
assert tres_centrales("Teror") == "ero" and tres_centrales("Tazacorte") == "aco"
assert insertar_en_centro("cuatro", "--") == "cua--tro"
try:
    tres_centrales("Arucas")
except ValueError as e:
    print("Error:", e)

ero aco cua--tro
Error: 'Arucas' no tiene longitud impar de al menos 3


In [14]:
def minusculas_primero(s):
    return "".join(sorted(s, key=str.isupper))  # sorted es estable: conserva el orden dentro de cada grupo


def intercalar(a, b):
    return "".join(x + y for x, y in zip_longest(a, b, fillvalue=""))


def intercalar_extremos(a, b):
    return intercalar(a, b[::-1])


print(minusculas_primero("PyThOnIsLa"), intercalar("abc", "12345"), intercalar_extremos("abc", "123"))
s = "LaGuaGuaLLeGa"
assert minusculas_primero(s) == "".join(c for c in s if not c.isupper()) + "".join(c for c in s if c.isupper())
assert intercalar("abc", "12345") == "a1b2c345"
assert intercalar_extremos("abc", "123") == "a3b2c1"


def intercalar_bucle(a, b):
    salida = ""
    for i in range(max(len(a), len(b))):
        salida += a[i:i + 1] + b[i:i + 1]
    return salida


random.seed(3)
for _ in range(100):
    a = "".join(random.choices("xyz", k=random.randint(0, 6)))
    b = "".join(random.choices("123", k=random.randint(0, 6)))
    assert intercalar(a, b) == intercalar_bucle(a, b)

yhnsaPTOIL a1b2c345 a3b2c1


## 11. Capitalizar sin `title()` y palabras alfanuméricas

Escribir `capitalizar(texto)`, que ponga en mayúscula la primera letra de cada palabra (separadas por espacios) sin usar `title()` ni `capitalize()`, y comparar el resultado con `title()` en textos con apóstrofos y con números. Después, extraer las palabras que mezclan letras y dígitos.

In [15]:
def capitalizar(texto):
    salida, anterior = [], " "
    for c in texto:
        salida.append(c.upper() if anterior == " " else c.lower())
        anterior = c
    return "".join(salida)


for t in ["la casa de o'shea", "3er premio de la feria", "SAN bartolomé de tirajana"]:
    print(f"{capitalizar(t):<28} | {t.title()}")

t = "la guagua de las palmas"
assert capitalizar(t) == t.title() == " ".join(p[0].upper() + p[1:] for p in t.split())
assert "o'shea".title() == "O'Shea" and capitalizar("o'shea") == "O'shea"
assert "3er".title() == "3Er" and capitalizar("3er") == "3er"

La Casa De O'shea            | La Casa De O'Shea
3er Premio De La Feria       | 3Er Premio De La Feria
San Bartolomé De Tirajana    | San Bartolomé De Tirajana


In [16]:
def alfanumericas(texto):
    palabras = (p.strip(string.punctuation) for p in texto.split())
    return [p for p in palabras if any(c.isalpha() for c in p) and any(c.isdigit() for c in p)]


texto = "La línea L1 sale a las 7h30; el código ABC123 vale, pero 2024 y guagua no."
print(alfanumericas(texto))
assert alfanumericas(texto) == ["L1", "7h30", "ABC123"]
limpias = [p.strip(string.punctuation) for p in texto.split()]
assert alfanumericas(texto) == [p for p in limpias if re.search(r"\d", p) and re.search(r"[^\W\d_]", p)]

['L1', '7h30', 'ABC123']


## 12. Caracteres únicos, sin repetidos y primer no repetido

- Decidir si todos los caracteres de una cadena son distintos, con un conjunto y sin él.
- Quitar los caracteres repetidos conservando la primera aparición de cada uno.
- Devolver el primer carácter que aparece una sola vez (o `None`).

In [17]:
def todos_distintos(s):
    return len(set(s)) == len(s)


def todos_distintos_sin_set(s):
    ordenada = sorted(s)
    return all(x != y for x, y in zip(ordenada, ordenada[1:]))


def sin_repetidos(s):
    return "".join(dict.fromkeys(s))


def primer_no_repetido(s):
    cuenta = Counter(s)
    return next((c for c in s if cuenta[c] == 1), None)


for s in ["murciélago", "barranco", "aabbcc", "tenderete"]:
    print(f"{s:<11} distintos={todos_distintos(s)!s:<5} sin repetidos={sin_repetidos(s):<8} "
          f"primer único={primer_no_repetido(s)}")

random.seed(11)
for _ in range(500):
    s = "".join(random.choices("abcde", k=random.randint(0, 7)))
    assert todos_distintos(s) == todos_distintos_sin_set(s)
    vistos = ""
    for c in s:
        if c not in vistos:
            vistos += c
    assert sin_repetidos(s) == vistos
    unicos = [c for c in s if s.count(c) == 1]
    assert primer_no_repetido(s) == (unicos[0] if unicos else None)

murciélago  distintos=True  sin repetidos=murciélago primer único=m
barranco    distintos=False sin repetidos=barnco   primer único=b
aabbcc      distintos=False sin repetidos=abc      primer único=None
tenderete   distintos=False sin repetidos=tendr    primer único=n


## 13. Invertir palabras

Escribir `invertir_orden(frase)`, que invierta el orden de las palabras, e `invertir_cada_una(frase)`, que dé la vuelta a cada palabra manteniendo su posición. Comprobar que aplicar dos veces cualquiera de ellas devuelve la frase y que, con un solo espacio entre palabras, combinar las dos equivale a invertir la cadena entera.

In [18]:
def invertir_orden(frase):
    return " ".join(reversed(frase.split()))


def invertir_cada_una(frase):
    return " ".join(p[::-1] for p in frase.split())


frase = "el alisio trae la panza de burro"
print(invertir_orden(frase))
print(invertir_cada_una(frase))
assert invertir_orden(invertir_orden(frase)) == frase
assert invertir_cada_una(invertir_cada_una(frase)) == frase
assert invertir_orden(invertir_cada_una(frase)) == frase[::-1]
assert invertir_orden("  muchos   espacios ") == "espacios muchos"

burro de panza la trae alisio el
le oisila eart al aznap ed orrub


## 14. Fortaleza de una contraseña

Puntuar una contraseña de 0 a 5 sumando un punto por cada criterio que cumpla: al menos 8 caracteres, alguna minúscula, alguna mayúscula, algún dígito y algún símbolo (un carácter que no es letra, dígito ni espacio). Mostrar una tabla con la puntuación, la etiqueta (`muy débil`, `débil`, `media`, `fuerte`, `muy fuerte`) y los criterios que faltan.

In [19]:
CRITERIOS = {
    "8+ caracteres": lambda c: len(c) >= 8,
    "minúscula": lambda c: any(x.islower() for x in c),
    "mayúscula": lambda c: any(x.isupper() for x in c),
    "dígito": lambda c: any(x.isdigit() for x in c),
    "símbolo": lambda c: any(not x.isalnum() and not x.isspace() for x in c),
}
ETIQUETAS = ["muy débil", "muy débil", "débil", "media", "fuerte", "muy fuerte"]


def fortaleza(clave):
    faltan = [nombre for nombre, cumple in CRITERIOS.items() if not cumple(clave)]
    return len(CRITERIOS) - len(faltan), faltan


def fortaleza_re(clave):
    patrones = [r"^.{8,}$", r"[a-zñáéíóúü]", r"[A-ZÑÁÉÍÓÚÜ]", r"\d", r"[^\w\s]|_"]
    return sum(bool(re.search(p, clave)) for p in patrones)


claves = ["", "teide", "Teide", "Teide2024", "T3ide!", "Teide-2024", "guagua_amarilla"]
print(f"{'Contraseña':<18} {'Puntos':>6}  {'Nivel':<11} Faltan")
for clave in claves:
    puntos, faltan = fortaleza(clave)
    print(f"{clave!r:<18} {puntos:>6}  {ETIQUETAS[puntos]:<11} {', '.join(faltan)}")
    assert puntos == fortaleza_re(clave)

Contraseña         Puntos  Nivel       Faltan
''                      0  muy débil   8+ caracteres, minúscula, mayúscula, dígito, símbolo
'teide'                 1  muy débil   8+ caracteres, mayúscula, dígito, símbolo
'Teide'                 2  débil       8+ caracteres, dígito, símbolo
'Teide2024'             4  fuerte      símbolo
'T3ide!'                4  fuerte      8+ caracteres
'Teide-2024'            5  muy fuerte  
'guagua_amarilla'       3  media       mayúscula, dígito


## 15. Formato de salida

Practicar la minilengua de formato de `format` y las *f-strings*:

- Alinear una palabra a la izquierda, a la derecha y al centro en 12 posiciones, rellenando con `.`, `-` y `*`.
- Escribir un número con ceros a la izquierda, con separador de miles (y cambiarlo al formato español) y con dos decimales.
- Escribir un entero en binario, octal y hexadecimal con y sin prefijo, y comprobarlo contra `bin`, `oct` y `hex`.
- Imprimir una tabla de ventas con las columnas alineadas y una fila de totales.

In [20]:
palabra = "mojo"
print(f"[{palabra:.<12}] [{palabra:->12}] [{palabra:*^12}]")
assert f"{palabra:*^12}" == "****mojo****" and format(palabra, ".<6") == "mojo.."

n = 1234567.891
print(f"{42:05d} | {n:,.2f} | {n:_.2f} | {-7:+04d}")
en_espanol = f"{n:,.2f}".translate(str.maketrans(",.", ".,"))
print(en_espanol)
assert f"{42:05d}" == "00042" and en_espanol == "1.234.567,89"

[mojo........] [--------mojo] [****mojo****]
00042 | 1,234,567.89 | 1_234_567.89 | -007
1.234.567,89


In [21]:
x = 2026
print(f"{'base':<12}{'sin prefijo':>14}{'con prefijo':>16}")
for nombre, codigo, funcion in [("binario", "b", bin), ("octal", "o", oct), ("hexadecimal", "x", hex)]:
    sin, con = format(x, codigo), format(x, "#" + codigo)
    print(f"{nombre:<12}{sin:>14}{con:>16}")
    assert con == funcion(x) and int(sin, {"b": 2, "o": 8, "x": 16}[codigo]) == x
print(f"{x:#X}, {x:016b}, {x:#010x}")

base           sin prefijo     con prefijo
binario        11111101010   0b11111101010
octal                 3752          0o3752
hexadecimal            7ea           0x7ea
0X7EA, 0000011111101010, 0x000007ea


In [22]:
ventas = [("plátanos", 312, 1.35), ("papas negras", 85, 3.2), ("gofio", 140, 2.75), ("queso de flor", 12, 19.5)]
print(f"{'Producto':<15}{'Kilos':>7}{'€/kg':>8}{'Total':>11}")
print("-" * 41)
for producto, kilos, precio in ventas:
    print(f"{producto:<15}{kilos:>7}{precio:>8.2f}{kilos * precio:>11,.2f}")
print("-" * 41)
total = sum(k * p for _, k, p in ventas)
print(f"{'TOTAL':<30}{total:>11,.2f}")
assert round(total, 2) == 1312.20

Producto         Kilos    €/kg      Total
-----------------------------------------
plátanos           312    1.35     421.20
papas negras        85    3.20     272.00
gofio              140    2.75     385.00
queso de flor       12   19.50     234.00
-----------------------------------------
TOTAL                            1,312.20


## 16. Validar una expresión matemática

Una expresión simple tiene operandos (enteros o decimales, con signo opcional) y operadores (`+`, `-`, `*`, `/`, `%`) separados por espacios, alternándose y empezando y terminando por un operando. Escribir `es_expresion_valida(texto)` y probarla con casos válidos y no válidos. Para las válidas, comparar el resultado de `eval` con un evaluador propio que respete la prioridad de `*`, `/` y `%`.

In [23]:
NUMERO = re.compile(r"[+-]?\d+(\.\d+)?")
OPERADORES = {"+", "-", "*", "/", "%"}


def es_expresion_valida(texto):
    tokens = texto.split()
    if len(tokens) % 2 == 0:
        return False
    return all(NUMERO.fullmatch(t) if i % 2 == 0 else t in OPERADORES for i, t in enumerate(tokens))


def evaluar(texto):
    tokens = texto.split()
    # Primero se agrupan los términos que se multiplican; después se suman
    terminos, signos = [float(tokens[0])], []
    for op, num in zip(tokens[1::2], map(float, tokens[2::2])):
        match op:
            case "*": terminos[-1] *= num
            case "/": terminos[-1] /= num
            case "%": terminos[-1] %= num
            case _:
                signos.append(op)
                terminos.append(num)
    resultado = terminos[0]
    for op, t in zip(signos, terminos[1:]):
        resultado = resultado + t if op == "+" else resultado - t
    return resultado


casos = {
    "5 + 6 / 7 - 4": True, "-3.5 * 2 % 4": True, "42": True, "10 - -2 * 3": True,
    "5 + + 3": False, "5 +": False, "* 5": False, "5 ^ 2": False, "5+6": False, "3 . 2": False, "": False,
}
for texto, esperado in casos.items():
    valida = es_expresion_valida(texto)
    detalle = f"= {evaluar(texto):g}" if valida else ""
    print(f"{texto!r:<18} {'válida' if valida else 'no válida':<10} {detalle}")
    assert valida == esperado
    if valida:
        assert abs(evaluar(texto) - eval(texto)) < 1e-12

'5 + 6 / 7 - 4'    válida     = 1.85714
'-3.5 * 2 % 4'     válida     = 1
'42'               válida     = 42
'10 - -2 * 3'      válida     = 16
'5 + + 3'          no válida  
'5 +'              no válida  
'* 5'              no válida  
'5 ^ 2'            no válida  
'5+6'              no válida  
'3 . 2'            no válida  
''                 no válida  
